# BinX AI & ML Internship
## Week 5 — Day 2: DBSCAN & Hierarchical Clustering

**Topics covered:**
- Why K-Means isn't always the right choice
- DBSCAN: density-based clustering and noise detection
- DBSCAN parameters: eps and min_samples
- Hierarchical clustering and dendrograms
- Choosing the right clustering method per situation
- Comparing all three methods on the same dataset

---

## 1. Where I'm Starting From

In Day 1, K-Means discovered four meaningful groups
in the insurance dataset.

After examining the characteristics of each group,
we interpreted one of them as a high-cost smoker cluster —
even though no label explicitly told K-Means to find such a group.

This is an important property of unsupervised learning:
the algorithm discovers the grouping, and we interpret
what those groups represent afterward.

But K-Means has real limitations. Before choosing a clustering
algorithm, it's important to understand what each one assumes
about the shape and structure of the data — and when those
assumptions break.

---

## 2. Limitations of K-Means

K-Means works best when clusters are relatively compact,
roughly spherical, and reasonably similar in size, and when
a suitable value of `k` can be chosen in advance.

When these conditions do not match the structure of the data,
K-Means can produce misleading clusters.

**Limitation 1 — `k` must be chosen in advance:**

```text
K-Means: "How many clusters?" → You must choose k before fitting.
DBSCAN:  "How many clusters?" → The number of clusters emerges
                                from the density structure of the data. 
  ```                              
K-Means does not discover the number of clusters by itself.
We use methods such as the Elbow Method and Silhouette Score
to help choose k.

DBSCAN does not require us to specify the number of clusters.
Instead, its result depends on density parameters such as
eps and min_samples.

Limitation 2 — Works best with compact, roughly spherical clusters:

K-Means assigns every point to its nearest centroid.
This works well when clusters are reasonably compact and
centered around distinct locations, but it can struggle with
irregularly shaped clusters:
```

Irregular data:        K-Means may struggle:



   ●●●                    ●● ●
 ●     ●                ● ●●● ●
●       ●               ● ● ●
 ●     ●                 ● ●
   ●●●
```

The problem is that K-Means tries to divide the data according
to distances from centroids, which may not match the actual
shape of an irregular cluster.

Limitation 3 — Forces every point into a cluster:

K-Means assigns every point to one of the clusters.
It has no built-in concept of "noise" or "doesn't belong anywhere."

Dense group 1  ●●●●●          ← cluster 1

Dense group 2  ●●●●●          ← cluster 2

Outlier            ●           ← K-Means still assigns it
                                  to the nearest cluster

A method such as DBSCAN can instead identify isolated points
as noise.

Two algorithms address different weaknesses of K-Means:

DBSCAN — useful for irregular shapes and noise/outliers
Hierarchical Clustering — reveals nested structure through
a dendrogram without requiring k in advance


---

## 3. DBSCAN — Density-Based Spatial Clustering

DBSCAN groups points that are **packed closely together**
and labels points in sparse regions as **noise**.

Unlike K-Means:
- It does **not** need k in advance
- It can find clusters of **any shape**
- It explicitly identifies **outliers** rather than forcing them into a cluster

### How DBSCAN Works

DBSCAN uses two parameters to define what "dense" means:

| Parameter | Controls |
|-----------|---------|
| `eps` | How close two points must be to count as neighbors |
| `min_samples` | How many neighbors a point needs to start a dense cluster |

The algorithm classifies every point into one of three types:

```text
Core point   → has at least min_samples neighbors within eps distance
               → can start or expand a cluster

Border point → within eps of a core point, but fewer than min_samples neighbors
               → belongs to the cluster but doesn't expand it

Noise point  → not within eps of any core point
               → labeled as -1 (outlier)
```

The full process:

```text
1. Pick an unvisited point
2. Count neighbors within eps distance
3. If neighbors ≥ min_samples → core point → start/expand cluster
4. If neighbors < min_samples and near a core → border point
5. If not near any core → noise point (label = -1)
6. Repeat until all points are visited
```

### The Output

```python
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=1.0, min_samples=3)
labels = db.fit_predict(X_scaled)
# label -1 = noise / outlier
# label  0 = cluster 0
# label  1 = cluster 1
# ...
```

### Choosing eps and min_samples

There is no single rule. Common approaches:

- **eps:** plot the k-nearest-neighbor distance for each point
  (sorted). Look for the elbow — that distance is a good eps candidate.
- **min_samples:** a common starting point is `2 × n_features`.
  For 4 features, try min_samples=8.

The parameters interact — if eps is too small, everything becomes noise.
If eps is too large, everything merges into one cluster.

---

## 4. Hierarchical Clustering

Hierarchical clustering is a clustering method that builds a
**hierarchy (tree) of clusters**.

Instead of choosing the number of clusters first, it starts with
every data point as its own cluster and gradually merges the
closest clusters together.

**How It Works**

Imagine we have five points:

```text
A   B       C   D       E
```

At the beginning, every point is a separate cluster:

```

[A] [B] [C] [D] [E]
```

Then the algorithm repeatedly merges the closest points or clusters:
```

Step 1:  [AB] [C] [D] [E]


Step 2:  [AB] [CD] [E]


Step 3:  [ABCD] [E]


Step 4:  [ABCDE]
```
The process continues until all points belong to one large cluster.



**The Dendrogram**

The merging process is shown using a dendrogram.

A dendrogram is a tree-like diagram that shows:

- Which points or clusters were merged
- The order in which they were merged
- How far apart the clusters were when they merged

A low merge height means the groups were relatively close.
A high merge height means the groups were more different.

We can "cut" the dendrogram at a chosen height to obtain
the number of clusters we want.

```

Distance
   │
   │              ┌──────────────┐
   │          ┌───┘              │
   │      ┌───┘              ┌───┘
   │  ┌───┘                ┌──┘
   │  │                    │
   └──┴────────────────────┴──────
          ↑
       Cut here

```
**Why Use Hierarchical Clustering?**

- We do not need to choose k before building the hierarchy.
- The dendrogram helps us understand the structure of the data.
- We can choose the number of clusters after seeing the tree.
- It can reveal smaller groups inside larger groups.


**Linkage Method**

- When we have two groups of points, we need a way to decide
how close the two groups are.

- This is called the linkage method.

- One common method is Ward linkage.

- Ward tries to merge groups while keeping the resulting clusters
as compact as possible.

```python
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster


# Build the hierarchical clustering structure
Z = linkage(X_scaled, method="ward")


# Visualize the hierarchy
dendrogram(Z)


# Extract 4 clusters from the hierarchy
labels = fcluster(Z, t=4, criterion="maxclust")
```

**What Does the Code Do?**

```
X_scaled
    ↓
linkage()
    ↓
Build the hierarchy
    ↓
Z
    ↓
dendrogram()
    ↓
Visualize the tree
    ↓
fcluster()
    ↓
Get cluster labels
```

**Limitation**

Hierarchical clustering can become slow and memory-intensive
when the dataset becomes very large because it needs to build
the full hierarchy.

For very large datasets, K-Means or DBSCAN may be more practical,
depending on the dataset and its structure.

---

## 5. Choosing the Right Clustering Method

Different clustering algorithms make different assumptions
about the data.

There is no single algorithm that is best for every dataset.

| Method | Best When | Watch Out For |
|--------|-----------|---------------|
| **K-Means** | Clusters are roughly round and similarly sized, and `k` is known or can be estimated | Needs `k` and assigns every point to a cluster |
| **DBSCAN** | Clusters have irregular shapes and noise/outliers may be present | Sensitive to `eps` and `min_samples`; can struggle with different cluster densities |
| **Hierarchical** | You want to understand relationships between groups or use a dendrogram | Can become slow and memory-intensive on very large datasets |

### Questions to Ask

Before choosing a clustering method, think about the structure
of your data:

```text
Do I have an idea of the number of clusters?

    → K-Means can be a good choice.

Are the clusters irregular in shape or are there outliers?

    → DBSCAN may be a better choice.

Do I want to see how smaller groups combine into larger groups?

    → Hierarchical clustering is useful.

Are the clusters very different in density?

    → DBSCAN may struggle, so another method may be more appropriate.

```
**The Big Picture**

- K-Means  → Find groups around centroids


- DBSCAN → Find dense regions and identify noise


- Hierarchical → Build a tree showing how groups are formed

The best method depends on the shape, density, size, and structure
of the dataset — not just on the number of clusters.

---

The next step is to apply these concepts to the same dataset
used in Day 1.

Hands-On Lab

In the practical lab, I will:

Run DBSCAN and identify its clusters and noise points.
Build a hierarchical clustering dendrogram.
Choose a suitable cut and identify the resulting clusters.
Compare K-Means, DBSCAN, and Hierarchical Clustering
on the same dataset.
Analyze which method fits the data best and explain why.

The goal is not only to run the algorithms, but to understand
why their results are different and when each method is useful.